# 02613 Re-exam 2024 — Code Proofs
Each cell proves the answer to one question by running the actual computation.


In [ ]:
import numpy as np


## Q3 — Single-processor time from multi-processor time (F=0.8, T(4)=10min)
Answer: **T(1) = 25 minutes**


In [ ]:
F = 0.8
p = 4
Tp = 10  # minutes on 4 cores

S4 = 1 / ((1 - F) + F/p)
T1 = Tp * S4
print(f'S(4) = {S4}')      # 2.5
print(f'T(1) = {T1} min')  # 25 min


## Q4 — Effect of reducing serial time by 3 minutes
Answer: **New T(4) = 7 minutes**


In [ ]:
# T(p) = T_serial + T_parallel/p
# T(4) = 10 = T_serial + T_parallel/4
# If T_serial reduced by 3: new T(4) = 10 - 3 = 7
T4_original = 10
serial_reduction = 3
T4_new = T4_original - serial_reduction  # only serial part changed
print(f'New T(4) = {T4_new} min')  # 7
# Note: the parallel part T_parallel/p does NOT change


## Q5 — Zarr block shape for column access (1000×100000 float64)
Code accesses `x[:, i]` — entire columns. Answer: **C) 1000×100**


In [ ]:
# For column access x[:, i], the optimal chunk spans all rows but few columns
# Shape 1000×100: each chunk covers ALL 1000 rows for 100 columns
# → one column access = one chunk read

rows, cols = 1000, 100_000
chunk_shapes = [(100, 100), (1000, 1), (1000, 100), (100, 1000)]

for cs in chunk_shapes:
    # For x[:,i]: need all rows for column i
    # Number of chunk reads = how many chunks span all rows for one column
    chunks_per_col_access = rows // cs[0]
    print(f'Chunk {cs}: {chunks_per_col_access} chunk(s) to read one full column')
# Best: 1 chunk per column access = chunk height = 1000 (all rows)


## Q6 — Memory size of one 1000×100 float64 Zarr block
Answer: **800 KB**


In [ ]:
rows, cols = 1000, 100
bytes_per_element = 8  # float64
block_bytes = rows * cols * bytes_per_element
block_KB = block_bytes / 1024
print(f'{rows}×{cols} float64 block = {block_bytes:,} bytes = {block_KB:.0f} KB')  # 800 KB


## Q7 — Line profiler: what is n_steps? (loop body Hits=10000)
Answer: **n_steps = 10000**


In [ ]:
# Line profiler Hits column = how many times that line executed
# For a for loop body: Hits = number of iterations = n_steps
loop_body_hits = 10000
n_steps = loop_body_hits
print(f'n_steps = {n_steps}')  # 10000


## Q8 — FLOP/s from line profiler (5 FLOPs/iter, 10000 iters, 15ms total)
Answer: **B) 3.33×10^6 FLOP/s**


In [ ]:
flops_per_iter = 5
n_iters        = 10_000
total_time_ms  = 15
total_time_s   = total_time_ms / 1000

total_flops = flops_per_iter * n_iters
flops_per_s = total_flops / total_time_s
print(f'Total FLOPs: {total_flops:,}')
print(f'FLOP/s:      {flops_per_s:.2e}')  # 3.33e6


## Q9 — NumPy expression for simulate (y accessed in reverse order)
Answer: **C) `np.sum((x*x+4) / (y[::-1]/x))`**


In [ ]:
# simulate: sum of (x[i]*x[i]+4) / (y[n-i-1]/x[i]) for i in range(n)
# y[n-i-1] = y reversed = y[::-1]

import numpy as np

np.random.seed(42)
n = 5
x = np.random.rand(n) + 0.1
y = np.random.rand(n) + 0.1

# Loop version
loop_result = sum((x[i]*x[i]+4) / (y[n-i-1]/x[i]) for i in range(n))

# NumPy version
numpy_result = np.sum((x*x + 4) / (y[::-1] / x))

print(f'Loop result:  {loop_result:.6f}')
print(f'NumPy result: {numpy_result:.6f}')
print(f'Match: {np.isclose(loop_result, numpy_result)}')  # True


## Q10 — Broadcasting shape (100×1×6×3) + (100×1×3)
Answer: **A) 100×100×6×3**


In [ ]:
a = np.ones((100, 1, 6, 3))
b = np.ones((100, 1, 3))

result = a + b
print(f'a shape: {a.shape}')
print(f'b shape: {b.shape}')
print(f'result:  {result.shape}')  # (100, 100, 6, 3)
# Step-by-step: right-align: a=(100,1,6,3), b=(1,100,1,3) after padding
# dim0: 100 vs 1 → 100
# dim1: 1   vs 100 → 100
# dim2: 6   vs 1 → 6
# dim3: 3   vs 3 → 3


## Q13 — How many HtoD/DtoH transfers (loop 100 images, NumPy arrays)?
Answer: **200 HtoD + 200 DtoH** (Numba auto-transfers ALL args both ways)


In [ ]:
# Numba rule: each kernel call with k NumPy args → k HtoD + k DtoH
n_iterations = 100
n_numpy_args = 2  # x (image) and y (output)

htod = n_iterations * n_numpy_args
dtoh = n_iterations * n_numpy_args
print(f'HtoD transfers: {htod}')  # 200
print(f'DtoH transfers: {dtoh}')  # 200
print(f'Total:          {htod + dtoh}')  # 400


## Q14 — Minimum (optimal) transfer count
Answer: **100 HtoD + 1 DtoH = 101 total**


In [ ]:
n_iterations = 100
# Optimal: transfer each image once HtoD; allocate output on device
# Only bring final result back with 1 DtoH
optimal_htod = n_iterations  # one per image input
optimal_dtoh = 1             # only final result back
print(f'Optimal HtoD: {optimal_htod}')
print(f'Optimal DtoH: {optimal_dtoh}')
print(f'Optimal total: {optimal_htod + optimal_dtoh}')  # 101


## Q18 — Parallel reduction speedup: row sums vs binary tree
Parallel row sums time = 2n; binary tree time = 2·log2(n). Speedup = n/log2(n)


In [ ]:
import numpy as np

# For n×n matrix:
# Row sums approach: parallel row sums (n ops with unlimited cores) + serial sum of n results
# Time ≈ n (parallel row sums) + n (serial combine) = 2n
# Binary tree: ceil(log2(n^2)) = 2*log2(n) rounds
# Speedup = 2n / (2*log2(n)) = n / log2(n)

for n in [10, 100, 1000]:
    row_sum_time  = 2 * n
    tree_time     = 2 * np.log2(n)
    speedup       = n / np.log2(n)
    print(f'n={n:4d}: row_sum_time={row_sum_time:6.0f}, tree_time={tree_time:5.1f}, speedup={speedup:.1f}x')
